# Option Metrics Calculator

Loads option chain data from the **SQLite snapshot DB** (or a fallback CSV) and computes practical trading metrics:
moneyness, break-even, intrinsic/extrinsic, probability of ITM, probability of profit (mid and ask-adjusted), and Black-Scholes Greeks.

Visualisation cells at the end show:
- Probability panel: P(ITM) vs P(profit at mid) vs P(profit at ask)
- Greeks vs strike
- Volatility smile
- Payoff diagram
- Breeden-Litzenberger market-implied PDF

## Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from skew import (
    compute_option_metrics,
    portfolio_greeks,
    market_implied_pdf,
    prob_profit_from_pdf,
    load_option_snapshots,
    list_snapshots,
)

In [ ]:
DB_PATH      = "../data/options.db"     # SQLite snapshot store
TICKER       = "NVDA"                   # which underlying to analyse
# Set snapshot_date to a specific "YYYY-MM-DD" to load that day's data,
# or leave as None to load the most recent available snapshot.
SNAPSHOT_DATE = None
DEFAULT_RF    = 0.04
CONTRACT_SIZE = 100

In [ ]:
raw = load_option_snapshots(TICKER, db_path=DB_PATH, snapshot_date=SNAPSHOT_DATE)

if raw.empty:
    raise RuntimeError(
        f"No data for {TICKER} in {DB_PATH}. "
        "Run fetch_opt_data.ipynb first to populate the DB."
    )

# Use the most recent snapshot date if none specified
snap_date = SNAPSHOT_DATE or raw["snapshot_date"].max()
df = raw[raw["snapshot_date"] == snap_date].copy()
print(f"Loaded {len(df)} rows  |  ticker={TICKER}  |  snapshot={snap_date}")
df.head(3)

In [ ]:
metrics = compute_option_metrics(df, default_rf=DEFAULT_RF, contract_size=CONTRACT_SIZE)

display_cols = [
    "option_type", "strike", "T", "implied_vol",
    "intrinsic", "extrinsic", "moneyness", "break_even",
    "prob_itm", "prob_profit", "prob_profit_ask",
    "delta", "gamma", "vega", "theta_day",
]
available = [c for c in display_cols if c in metrics.columns]
metrics[available].sort_values(["option_type", "strike"]).head(20)

In [ ]:
port = portfolio_greeks(metrics)
pd.Series(port).rename("net_position").to_frame()

In [ ]:
# ── Probability panel: P(ITM) vs P(profit at mid) vs P(profit at ask) ──
# Pick the nearest expiry slice for a clean single-surface view
nearest_T = metrics["T"].min()
slice_df = metrics[np.isclose(metrics["T"], nearest_T, atol=1e-3)].copy()

calls = slice_df[slice_df["option_type"] == "C"].sort_values("strike")
puts  = slice_df[slice_df["option_type"] == "P"].sort_values("strike")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle(f"{TICKER}  |  nearest expiry  |  T≈{nearest_T*365:.0f}d", fontsize=13)

for ax, side, label in zip(axes, [calls, puts], ["Calls", "Puts"]):
    ax.plot(side["strike"], side["prob_itm"],        label="P(ITM)",          lw=2)
    ax.plot(side["strike"], side["prob_profit"],      label="P(profit @ mid)", lw=2, linestyle="--")
    ax.plot(side["strike"], side["prob_profit_ask"],  label="P(profit @ ask)", lw=2, linestyle=":")
    ax.axvline(side["spot"].iloc[0], color="grey", lw=1, linestyle="--", alpha=0.6, label="Spot")
    ax.set_title(label)
    ax.set_xlabel("Strike")
    ax.set_ylabel("Probability")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Greeks vs strike (nearest expiry, calls) ──
calls_near = slice_df[slice_df["option_type"] == "C"].sort_values("strike")

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle(f"{TICKER} Call Greeks  |  T≈{nearest_T*365:.0f}d", fontsize=13)

greek_specs = [
    ("delta",     "Delta",         "tab:blue"),
    ("gamma",     "Gamma",         "tab:orange"),
    ("vega",      "Vega (÷100)",   "tab:green"),
    ("theta_day", "Theta/day",     "tab:red"),
    ("rho",       "Rho (÷100)",    "tab:purple"),
    ("prob_itm",  "P(ITM)",        "tab:brown"),
]

for ax, (col, title, color) in zip(axes.flat, greek_specs):
    if col in calls_near.columns:
        ax.plot(calls_near["strike"], calls_near[col], color=color, lw=2)
        ax.axvline(calls_near["spot"].iloc[0], color="grey", lw=1, linestyle="--", alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel("Strike")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Volatility smile / surface ──
iv_col = "implied_vol" if "implied_vol" in metrics.columns else "iv"

fig, ax = plt.subplots(figsize=(12, 5))
expiries = sorted(metrics["T"].unique())
cmap = plt.cm.viridis(np.linspace(0, 1, len(expiries)))

for t_val, color in zip(expiries, cmap):
    sub = metrics[np.isclose(metrics["T"], t_val, atol=1e-3)]
    calls_sub = sub[sub["option_type"] == "C"].sort_values("strike")
    if len(calls_sub) > 2:
        ax.plot(calls_sub["strike"], calls_sub[iv_col] * 100, color=color,
                alpha=0.7, lw=1.5, label=f"T≈{t_val*365:.0f}d")

spot_val = metrics["spot"].iloc[0]
ax.axvline(spot_val, color="black", lw=1, linestyle="--", alpha=0.5, label="Spot")
ax.set_title(f"{TICKER} Implied Volatility Smile")
ax.set_xlabel("Strike")
ax.set_ylabel("IV (%)")
ax.legend(fontsize=7, ncol=4, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Breeden-Litzenberger market-implied PDF (nearest expiry) ──
# Rename columns so market_implied_pdf finds what it needs
_chain = slice_df.rename(columns={"lastPrice": "option_price", "implied_vol": "iv"}).copy()
if "option_price" not in _chain.columns and "iv" not in _chain.columns:
    # fallback to original column names
    _chain = slice_df.copy()

try:
    K_fine, pdf = market_implied_pdf(
        _chain,
        spot=spot_val,
        r=DEFAULT_RF,
        T=nearest_T,
        price_col="option_price",
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(K_fine, pdf, alpha=0.25, color="steelblue", label="Market-implied PDF")
    ax.plot(K_fine, pdf, color="steelblue", lw=2)
    ax.axvline(spot_val, color="grey", lw=1.5, linestyle="--", label=f"Spot {spot_val:.1f}")

    # Shade P(profit) region for an ATM call as an example
    atm_call = calls_near.iloc[(calls_near["strike"] - spot_val).abs().argsort()[:1]]
    if not atm_call.empty:
        be = atm_call["break_even"].iloc[0]
        be_ask = float(atm_call["strike"].iloc[0] + atm_call["ask"].iloc[0]) if "ask" in atm_call.columns else be
        p_mid = prob_profit_from_pdf(K_fine, pdf, be, "C")
        p_ask = prob_profit_from_pdf(K_fine, pdf, be_ask, "C")
        mask_mid = K_fine >= be
        ax.fill_between(K_fine[mask_mid], pdf[mask_mid], alpha=0.35, color="green",
                        label=f"P(profit@mid)={p_mid:.1%}  BE={be:.1f}")
        mask_ask = K_fine >= be_ask
        ax.fill_between(K_fine[mask_ask], pdf[mask_ask], alpha=0.35, color="orange",
                        label=f"P(profit@ask)={p_ask:.1%}  BE={be_ask:.1f}")

    ax.set_title(f"{TICKER} Market-Implied PDF  |  T≈{nearest_T*365:.0f}d")
    ax.set_xlabel("Strike at expiry")
    ax.set_ylabel("Risk-neutral density")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"PDF fitting skipped: {e}")

In [ ]:
# ── ATM call & put payoff diagrams at expiry ──
atm_call_row = calls_near.iloc[(calls_near["strike"] - spot_val).abs().argsort()[:1]].iloc[0]
atm_put_row  = slice_df[slice_df["option_type"] == "P"].sort_values("strike").iloc[
    (slice_df[slice_df["option_type"] == "P"]["strike"] - spot_val).abs().argsort()[:1]
].iloc[0]

K_c   = atm_call_row["strike"];  prem_c = atm_call_row["lastPrice"] if "lastPrice" in atm_call_row else atm_call_row.get("option_price", 0)
K_p   = atm_put_row["strike"];   prem_p = atm_put_row["lastPrice"]  if "lastPrice" in atm_put_row  else atm_put_row.get("option_price", 0)

S_range = np.linspace(spot_val * 0.6, spot_val * 1.4, 400)

pnl_call = np.maximum(S_range - K_c, 0) - prem_c
pnl_put  = np.maximum(K_p - S_range, 0) - prem_p

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{TICKER} ATM Option Payoff at Expiry  |  Spot={spot_val:.1f}", fontsize=13)

for ax, pnl, label, K, prem, color in [
    (axes[0], pnl_call, f"ATM Call  K={K_c:.0f}  prem={prem_c:.2f}", K_c, prem_c, "tab:blue"),
    (axes[1], pnl_put,  f"ATM Put   K={K_p:.0f}  prem={prem_p:.2f}", K_p, prem_p, "tab:red"),
]:
    ax.plot(S_range, pnl, color=color, lw=2, label=label)
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(spot_val, color="grey", lw=1, linestyle="--", alpha=0.6, label="Spot today")
    ax.fill_between(S_range, pnl, 0, where=(pnl > 0), alpha=0.2, color="green", label="Profit zone")
    ax.fill_between(S_range, pnl, 0, where=(pnl < 0), alpha=0.2, color="red",   label="Loss zone")
    ax.set_xlabel("Underlying price at expiry")
    ax.set_ylabel("P&L per share")
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()